In [16]:
import numpy as np 
import torch 
import torch.nn as nn 
import matplotlib.pyplot as plt 
import gymnasium as gym 
from config import *

In [17]:
import random
import numpy as np
import torch

def set_seed(seed=44):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True)

set_seed(44)

In [ ]:
from dataclasses import dataclass
import numpy as np

@dataclass
class UAV:
    id: int
    position: np.ndarray
    velocity: np.ndarray
    battery_j: float

    active: bool = True

@dataclass
class Target:
    id: int
    position: np.ndarray
    confirmed: bool = False

@dataclass
class Obstacle:
    position: np.ndarray
    radius: float
    height: float

    def __post_init__(self):
        self.position = np.asarray(
            self.position,
            dtype=np.float64,
        )
        self.radius = float(self.radius)
        self.height = float(self.height)

        if self.position.shape != (2,):
            raise ValueError(
                "Obstacle.position must be the XY center with shape (2,)"
            )

        if not np.all(np.isfinite(self.position)):
            raise ValueError(
                "Obstacle.position must contain only finite values"
            )

        if not np.isfinite(self.radius) or self.radius <= 0.0:
            raise ValueError(
                "Obstacle.radius must be finite and > 0"
            )

        if not np.isfinite(self.height) or self.height <= 0.0:
            raise ValueError(
                "Obstacle.height must be finite and > 0"
            )

@dataclass
class Report:
    target_id: int
    source_uav: int
    created_step: int
    size_bytes: int
    ttl_s: float
    delivered_bytes: int = 0

    def __post_init__(self):
        for name, value in (
            ("target_id", self.target_id),
            ("source_uav", self.source_uav),
            ("created_step", self.created_step),
            ("size_bytes", self.size_bytes),
            ("delivered_bytes", self.delivered_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(value, (int, np.integer))
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        self.target_id = int(self.target_id)
        self.source_uav = int(self.source_uav)
        self.created_step = int(self.created_step)
        self.size_bytes = int(self.size_bytes)
        self.delivered_bytes = int(self.delivered_bytes)
        self.ttl_s = float(self.ttl_s)

        if self.target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if self.source_uav < 0:
            raise ValueError(
                "source_uav must be >= 0"
            )

        if self.created_step < 0:
            raise ValueError(
                "created_step must be >= 0"
            )

        if self.size_bytes <= 0:
            raise ValueError(
                "size_bytes must be > 0"
            )

        if (
            not np.isfinite(self.ttl_s)
            or self.ttl_s <= 0.0
        ):
            raise ValueError(
                "ttl_s must be finite and > 0"
            )

        if not (
            0
            <= self.delivered_bytes
            <= self.size_bytes
        ):
            raise ValueError(
                "delivered_bytes must be in "
                "[0, size_bytes]"
            )


In [19]:
def create_uav(rng):
    if not isinstance(rng, np.random.Generator):
        raise TypeError("rng must be numpy.random.Generator")

    num_uavs = int(CONFIG["num_uavs"])
    gcs = np.asarray(CONFIG["gcs_position"], dtype=np.float64)
    launch_radius = float(CONFIG["launch_radius_m"])
    safety_distance = float(CONFIG["safety_distance"])
    launch_min_spacing = float(CONFIG["launch_min_spacing_m"])
    map_size = float(CONFIG["map_size"])
    altitude = float(CONFIG["altitude_min"])

    if num_uavs < 1:
        raise ValueError("num_uavs must be >= 1")

    if gcs.shape != (3,) or not np.all(np.isfinite(gcs)):
        raise ValueError("gcs_position must be a finite 3D position")

    if not (
        0.0 <= gcs[0] <= map_size
        and 0.0 <= gcs[1] <= map_size
    ):
        raise ValueError("gcs_position must lie inside the map")

    if not np.isfinite(launch_radius) or launch_radius <= 0.0:
        raise ValueError("launch_radius_m must be finite and > 0")

    if not np.isfinite(safety_distance) or safety_distance < 0.0:
        raise ValueError("safety_distance must be finite and >= 0")

    if (
        not np.isfinite(launch_min_spacing)
        or launch_min_spacing < safety_distance
    ):
        raise ValueError(
            "launch_min_spacing_m must be finite and >= safety_distance"
        )

    if not np.isfinite(map_size) or map_size <= 0.0:
        raise ValueError("map_size must be finite and > 0")

    if not np.isfinite(altitude):
        raise ValueError("altitude_min must be finite")

    uavs = []
    max_attempts = max(10_000, 1_000 * num_uavs)
    attempts = 0
    launch_radius_sq = launch_radius * launch_radius

    while len(uavs) < num_uavs:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all UAVs inside the launch region while "
                "respecting map bounds and launch_min_spacing_m. "
                "Increase launch_radius_m or reduce num_uavs/launch_min_spacing_m."
            )

        offset_xy = rng.uniform(
            -launch_radius,
            launch_radius,
            size=2,
        ).astype(np.float64)

        if float(offset_xy @ offset_xy) > launch_radius_sq:
            continue

        candidate_xy = gcs[:2] + offset_xy

        if not (
            0.0 <= candidate_xy[0] <= map_size
            and 0.0 <= candidate_xy[1] <= map_size
        ):
            continue

        too_close = any(
            np.linalg.norm(candidate_xy - uav.position[:2])
            < launch_min_spacing
            for uav in uavs
        )

        if too_close:
            continue

        position = np.array(
            [candidate_xy[0], candidate_xy[1], altitude],
            dtype=np.float64,
        )

        uavs.append(
            UAV(
                id=len(uavs),
                position=position,
                velocity=np.zeros(3, dtype=np.float64),
                battery_j=CONFIG["battery_j"],
                active=True,
            )
        )

    return uavs


In [ ]:
def point_inside_obstacle(point, obstacles, margin=0.0):
    point = np.asarray(point, dtype=np.float64)
    margin = float(margin)

    if not np.isfinite(margin) or margin < 0.0:
        raise ValueError(
            "margin must be finite and >= 0"
        )

    if point.shape == (2,):
        point_xy = point
        z = 0.0
    elif point.shape == (3,):
        point_xy = point[:2]
        z = float(point[2])
    else:
        raise ValueError(
            "point must have shape (2,) or (3,)"
        )

    if not np.all(np.isfinite(point)):
        raise ValueError(
            "point must contain only finite values"
        )

    for obs in obstacles:
        horizontal_distance = np.linalg.norm(
            point_xy - obs.position
        )

        inside_horizontal = (
            horizontal_distance
            <= obs.radius + margin
        )
        inside_vertical = (
            0.0 <= z <= obs.height + margin
        )

        if inside_horizontal and inside_vertical:
            return True

    return False


In [21]:
def create_obstacle(rng, uavs):
    obstacles = []

    map_size = float(CONFIG["map_size"])
    r_min = float(CONFIG["obstacle_radius_min_m"])
    r_max = float(CONFIG["obstacle_radius_max_m"])
    h_min = float(CONFIG["obstacle_height_min_m"])
    h_max = float(CONFIG["obstacle_height_max_m"])

    safety_margin = float(CONFIG["safety_distance"])
    gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
    gcs_exclusion = float(CONFIG["gcs_exclusion_radius_m"])

    if r_min <= 0.0 or r_max < r_min:
        raise ValueError("obstacle radius range is invalid")
    if 2.0 * r_max > map_size:
        raise ValueError(
            "obstacle_radius_max_m must be <= map_size / 2"
        )

    max_attempts = 100000
    attempts = 0

    while len(obstacles) < CONFIG["num_obstacles"]:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all obstacles with the current constraints."
            )

        radius = float(rng.uniform(r_min, r_max))
        height = float(rng.uniform(h_min, h_max))

        x = float(rng.uniform(radius, map_size - radius))
        y = float(rng.uniform(radius, map_size - radius))
        center = np.array([x, y], dtype=np.float64)

        distance_to_gcs = np.linalg.norm(center - gcs_xy)
        if distance_to_gcs <= gcs_exclusion + radius:
            continue

        overlaps_launch = any(
            np.linalg.norm(center - uav.position[:2])
            <= radius + safety_margin
            for uav in uavs
        )
        if overlaps_launch:
            continue

        overlaps_obstacle = any(
            np.linalg.norm(center - other.position)
            <= radius + other.radius
            for other in obstacles
        )
        if overlaps_obstacle:
            continue

        obstacles.append(
            Obstacle(
                position=center,
                radius=radius,
                height=height,
            )
        )

    return obstacles


In [22]:
def create_target(rng, obstacles):
    targets = []
    occupied_cells = set()

    map_size = float(CONFIG["map_size"])
    cell_size = float(CONFIG["grid_cell_m"])
    gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
    target_exclusion = float(CONFIG["target_exclusion_radius_m"])

    max_attempts = 100000
    attempts = 0

    while len(targets) < CONFIG["num_targets"]:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all targets with the current constraints."
            )

        position = rng.uniform(0.0,map_size,size=2,).astype(np.float64)

        if point_inside_obstacle(position, obstacles):
            continue

        if np.linalg.norm(position - gcs_xy) <= target_exclusion:
            continue

        gx = int(np.floor(position[0] / cell_size))
        gy = int(np.floor(position[1] / cell_size))
        cell = (gx, gy)

        if cell in occupied_cells:
            continue

        targets.append(
            Target(
                id=len(targets),
                position=position,
                confirmed=False,
            )
        )
        occupied_cells.add(cell)

    return targets


In [23]:
def create_world(seed):
    rng = np.random.default_rng(seed)
    uavs = create_uav(rng)
    obstacles = create_obstacle(rng, uavs)
    targets = create_target(rng, obstacles)

    return rng, uavs, targets, obstacles


In [ ]:
def segment_intersects_obstacle(
    start_position,
    end_position,
    obstacles,
    margin=0.0,
):
    start = np.asarray(
        start_position,
        dtype=np.float64,
    )

    end = np.asarray(
        end_position,
        dtype=np.float64,
    )

    if start.shape != (3,) or end.shape != (3,):
        raise ValueError(
            "start_position and end_position must have shape (3,)"
        )

    if (
        not np.all(np.isfinite(start))
        or not np.all(np.isfinite(end))
    ):
        raise ValueError(
            "start_position and end_position "
            "must contain only finite values"
        )

    margin = float(margin)

    if not np.isfinite(margin) or margin < 0.0:
        raise ValueError(
            "margin must be finite and >= 0"
        )

    direction = end - start
    eps = 1e-12

    for obs in obstacles:

        center = np.asarray(obs.position,dtype=np.float64,)

        radius = float(obs.radius) + float(margin)
        height = float(obs.height) + float(margin)

        dz = float(direction[2])

        if abs(dz) <= eps:

            if not (
                0.0 <= start[2] <= height
            ):
                continue

            z_enter = 0.0
            z_exit = 1.0

        else:

            t_ground = (0.0 - start[2]) / dz
            t_top = (height - start[2]) / dz

            z_enter = max(0.0,min(t_ground, t_top))
            z_exit = min(1.0,max(t_ground, t_top))

            if z_enter > z_exit:
                continue

        relative_xy = start[:2] - center[:2]
        direction_xy = direction[:2]

        a = direction_xy@direction_xy
        c = (relative_xy@relative_xy)- radius * radius

        if a <= eps:

            if c > 0:
                continue

            xy_enter = 0
            xy_exit = 1

        else:

            b = 2 * relative_xy@direction_xy
            discriminant = (b * b- 4.0 * a * c)

            if discriminant < 0.0:
                continue

            root = np.sqrt(max(discriminant, 0.0))

            t1 = (-b - root) / (2.0 * a)
            t2 = (-b + root) / (2.0 * a)

            xy_enter = max(0.0,min(t1, t2))
            xy_exit = min(1.0,max(t1, t2))

            if xy_enter > xy_exit:
                continue

        enter = max(z_enter,xy_enter)
        exit_ = min(z_exit,xy_exit)

        if enter <= exit_:
            return True

    return False


In [ ]:
def compute_motion_candidate(
    uav,
    action,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    action = np.asarray(
        action,
        dtype=np.float64,
    )

    if action.shape != (3,):
        raise ValueError("action must have shape (3,)")

    if not np.all(np.isfinite(action)):
        raise ValueError("action must contain only finite values")

    if not uav.active:
        return (
            uav.position.copy(),
            np.zeros(3, dtype=np.float64),
            False,
        )

    action = np.clip(
        action,
        -1.0,
        1.0,
    )

    action_norm = np.linalg.norm(action)

    if action_norm > 1.0:
        action = action / action_norm

    acceleration = action * CONFIG["max_accel"]

    candidate_velocity = (
        uav.velocity
        + acceleration * dt
    )

    speed = np.linalg.norm(candidate_velocity)

    if speed > CONFIG["max_speed"]:
        candidate_velocity = (candidate_velocity/ speed* CONFIG["max_speed"])

    old_position = uav.position.copy()

    raw_candidate_position = old_position+ candidate_velocity * dt

    candidate_position = raw_candidate_position.copy()

    candidate_position[0] = np.clip(
        candidate_position[0],
        0.0,
        CONFIG["map_size"],
    )

    candidate_position[1] = np.clip(
        candidate_position[1],
        0.0,
        CONFIG["map_size"],
    )

    candidate_position[2] = np.clip(
        candidate_position[2],
        CONFIG["altitude_min"],
        CONFIG["altitude_max"],
    )

    boundary_clipped = not np.allclose(
        raw_candidate_position,
        candidate_position,
    )

    actual_velocity = (
        candidate_position
        - old_position
    ) / dt

    return (
        candidate_position,
        actual_velocity,
        boundary_clipped,
    )


In [26]:
def minimum_distance_during_motion(
    start_a,
    end_a,
    start_b,
    end_b,
):
    start_a = np.asarray(start_a,dtype=np.float64,)
    end_a = np.asarray(end_a,dtype=np.float64,)
    start_b = np.asarray(start_b,dtype=np.float64,)
    end_b = np.asarray(end_b,dtype=np.float64,)
    for point in (
        start_a,
        end_a,
        start_b,
        end_b,
    ):
        if point.shape != (3,):
            raise ValueError(
                "all positions must have shape (3,)"
            )

        if not np.all(np.isfinite(point)):
            raise ValueError(
                "all positions must contain only finite values"
            )
    relative_start = start_a - start_b
    displacement_a = end_a - start_a
    displacement_b = end_b - start_b

    relative_motion = displacement_a - displacement_b

    denominator = relative_motion@relative_motion
    if denominator <= 1e-12:
        return np.linalg.norm(
                relative_start
            )
        

    t_closest = -(relative_start@relative_motion)/ denominator
    t_closest = np.clip(t_closest,0.0,1.0,)

    relative_at_closest = (
        relative_start
        + t_closest * relative_motion
    )

    return float(
        np.linalg.norm(
            relative_at_closest
        )
    )

In [27]:
def apply_swarm_motion(
    uavs,
    actions,
    obstacles=None,
    dt=None,
):
    if obstacles is None:
        obstacles = []

    if dt is None:
        dt = CONFIG["dt"]
    dt = float(dt)


    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    num_uavs = len(uavs)

    if num_uavs == 0:
        raise ValueError(
            "uavs must not be empty"
        )

    actions = np.asarray(actions,dtype=np.float64)

    if actions.shape != (num_uavs, 3):
        raise ValueError(
            f"actions must have shape "
            f"({num_uavs}, 3)"
        )

    if not np.all(np.isfinite(actions)):
        raise ValueError(
            "actions must contain only finite values"
        )

    old_positions = np.stack([
        uav.position.copy()
        for uav in uavs
    ])

    safety_distance = float(CONFIG["safety_distance"])

    for i in range(num_uavs):
        if not uavs[i].active:
            continue

        for j in range(i + 1,num_uavs):
            if not uavs[j].active:
                continue

            distance = np.linalg.norm(old_positions[i]- old_positions[j])

            if distance < safety_distance:
                raise ValueError(
                    "initial active UAV positions "
                    "violate safety_distance"
                )

    candidate_positions = []
    candidate_velocities = []
    boundary_clipped = np.zeros(
        num_uavs,
        dtype=bool,
    )

    for i, (uav, action) in enumerate(zip(uavs, actions)):
        
        position,velocity,clipped= compute_motion_candidate(
            uav,
            action,
            dt=dt,
        )

        candidate_positions.append(position)
        candidate_velocities.append(velocity)

        boundary_clipped[i] = clipped

    candidate_positions = np.stack(candidate_positions)
    candidate_velocities = np.stack(candidate_velocities)

    blocked_by_obstacle = np.zeros(
        num_uavs,
        dtype=bool,
    )

    for i, uav in enumerate(uavs):
        if not uav.active:
            continue

        if segment_intersects_obstacle(
            old_positions[i],
            candidate_positions[i],
            obstacles,
        ):
            blocked_by_obstacle[i] = True

    blocked_by_peer = np.zeros(num_uavs,dtype=bool)
    blocked = blocked_by_obstacle.copy()
    

    while True:
        effective_positions = candidate_positions.copy()
        effective_positions[blocked] = old_positions[blocked]

        next_blocked = blocked.copy()

        for i in range(num_uavs):
            if not uavs[i].active:
                continue

            for j in range(i + 1,num_uavs):
                if not uavs[j].active:
                    continue

                distance = minimum_distance_during_motion(
                        old_positions[i],
                        effective_positions[i],
                        old_positions[j],
                        effective_positions[j],
                    )
                
                if distance < safety_distance:
                    next_blocked[i] = True
                    next_blocked[j] = True

                    blocked_by_peer[i] = True
                    blocked_by_peer[j] = True

        if np.array_equal(next_blocked,blocked):
            break

        blocked = next_blocked

    for i, uav in enumerate(uavs):
        if not uav.active:
            uav.velocity = np.zeros(3, dtype=np.float64)
            continue

        if blocked[i]:
            uav.position = old_positions[i].copy()
            uav.velocity = np.zeros(3,dtype=np.float64)

        else:
            uav.position = candidate_positions[i].copy()
            uav.velocity = candidate_velocities[i].copy()

    return {
        "blocked": blocked,
        "blocked_by_obstacle":blocked_by_obstacle,
        "blocked_by_peer":blocked_by_peer,
        "boundary_clipped":boundary_clipped,
    }


In [ ]:
def sensing_profile(altitude):
    altitude = float(altitude)

    if not np.isfinite(altitude):
        raise ValueError("altitude must be finite")

    altitude = np.clip(altitude,CONFIG["altitude_min"],CONFIG["altitude_max"])
    altitude_anchors = np.asarray(CONFIG["altitude_anchors"])

    pd_anchors = np.asarray(CONFIG["pd"])
    pf_anchors = np.asarray(CONFIG["pf"])

    pd = np.interp(altitude,altitude_anchors,pd_anchors)
    pf = np.interp(altitude,altitude_anchors,pf_anchors)

    full_fov = CONFIG["camera_full_fov_deg"]
    half_fov_rad = np.deg2rad(full_fov / 2.0)
    fov_radius = altitude* np.tan(half_fov_rad)

    return (
        float(pd),
        float(pf),
        float(fov_radius)
    )

In [29]:
def create_belief_maps():
    grid_n = int(np.ceil(CONFIG["map_size"] / CONFIG["grid_cell_m"]))
    belief_maps = np.full((CONFIG["num_uavs"],grid_n,grid_n),CONFIG["belief_prior"])
    return belief_maps

In [ ]:
def world_to_grid(position_xy):
    position_xy = np.asarray(position_xy, dtype=np.float64)

    if position_xy.ndim != 1 or position_xy.size < 2:
        raise ValueError(
            "position_xy must be a 1D array containing at least x and y"
        )

    x = float(position_xy[0])
    y = float(position_xy[1])

    if not np.isfinite(x) or not np.isfinite(y):
        raise ValueError("position must be finite")

    map_size = float(CONFIG["map_size"])
    cell_size = float(CONFIG["grid_cell_m"])

    if not np.isfinite(map_size) or map_size <= 0.0:
        raise ValueError("map_size must be finite and > 0")

    if not np.isfinite(cell_size) or cell_size <= 0.0:
        raise ValueError("grid_cell_m must be finite and > 0")

    if not (
        0.0 <= x <= map_size
        and 0.0 <= y <= map_size
    ):
        raise ValueError("position is outside the map")

    grid_n = int(np.ceil(map_size / cell_size))

    gx = int(np.floor(x / cell_size))
    gy = int(np.floor(y / cell_size))

    gx = min(gx, grid_n - 1)
    gy = min(gy, grid_n - 1)

    return gx, gy


In [31]:
def cell_intersects_fov(
    gx,
    gy,
    uav_xy,
    fov_radius,
):
    cell_size = float(CONFIG["grid_cell_m"])
    map_size = float(CONFIG["map_size"])

    x_min = gx * cell_size
    x_max = min((gx + 1) * cell_size,map_size)

    y_min = gy * cell_size
    y_max = min((gy + 1) * cell_size,map_size)

    closest_x = np.clip(uav_xy[0],x_min,x_max)
    closest_y = np.clip(uav_xy[1],y_min,y_max)

    dx = float(uav_xy[0] - closest_x)
    dy = float(uav_xy[1] - closest_y)

    return (
        dx * dx + dy * dy <= fov_radius * fov_radius
    )

In [32]:
def cell_center_in_fov(gx, gy, uav_xy, fov_radius):
    cell_size = float(CONFIG["grid_cell_m"])
    map_size = float(CONFIG["map_size"])

    center_x = min((gx + 0.5) * cell_size, map_size)
    center_y = min((gy + 0.5) * cell_size, map_size)

    dx = float(center_x - uav_xy[0])
    dy = float(center_y - uav_xy[1])

    return dx * dx + dy * dy <= fov_radius * fov_radius


def cells_in_fov(uav):
    altitude = float(uav.position[2])

    if altitude <= 0.0:
        return []

    _, _, fov_radius = sensing_profile(altitude)

    cell_size = float(CONFIG["grid_cell_m"])
    grid_n = int(np.ceil(CONFIG["map_size"] / cell_size))

    uav_xy = np.asarray(uav.position[:2], dtype=np.float64)
    uav_x = float(uav_xy[0])
    uav_y = float(uav_xy[1])

    gx_min = max(0, int(np.floor((uav_x - fov_radius) / cell_size)))
    gx_max = min(grid_n - 1, int(np.floor((uav_x + fov_radius) / cell_size)))
    gy_min = max(0, int(np.floor((uav_y - fov_radius) / cell_size)))
    gy_max = min(grid_n - 1, int(np.floor((uav_y + fov_radius) / cell_size)))

    visible_cells = []

    for gy in range(gy_min, gy_max + 1):
        for gx in range(gx_min, gx_max + 1):
            if cell_center_in_fov(gx, gy, uav_xy, fov_radius):
                visible_cells.append((gx, gy))

    return visible_cells

In [33]:
def get_target_cells(targets):
    target_cells = {}

    for target in targets:
        gx, gy = world_to_grid(target.position)
        target_cells.setdefault((gx, gy), []).append(target.id)

    return target_cells


In [34]:
def sample_sensor_measurement(has_target,pd,pf,rng):
    if has_target:
        positive_probability = pd

    else:
        positive_probability = pf
    observation = (rng.random()< positive_probability)

    return int(observation)

In [ ]:
def bayes_update(prior, observation, pd, pf, eps=1e-8):
    eps = float(eps)

    if (
        not np.isfinite(eps)
        or not 0.0 < eps < 0.5
    ):
        raise ValueError(
            "eps must be finite and in (0, 0.5)"
        )

    prior = float(prior)
    pd = float(pd)
    pf = float(pf)

    for name, value in (
        ("prior", prior),
        ("pd", pd),
        ("pf", pf),
    ):
        if not np.isfinite(value):
            raise ValueError(
                f"{name} must be finite"
            )

        if not 0.0 <= value <= 1.0:
            raise ValueError(
                f"{name} must be in [0, 1]"
            )

    prior = np.clip(prior,eps,1.0 - eps)

    if observation == 1:
        numerator = pd * prior
        denominator = pd * prior + pf * (1.0 - prior)
    elif observation == 0:
        numerator = (1.0 - pd) * prior
        denominator = (
            (1.0 - pd) * prior
            + (1.0 - pf) * (1.0 - prior)
        )
    else:
        raise ValueError("observation must be 0 or 1")

    denominator = max(float(denominator), eps)
    posterior = numerator / denominator

    return float(np.clip(posterior, eps, 1.0 - eps))


In [36]:
def sense_and_update(uav, belief_map, targets, rng):
    altitude = float(uav.position[2])

    if altitude <= 0.0:
        return []

    pd, pf, fov_radius = sensing_profile(altitude)
    center_visible_cells = set(cells_in_fov(uav))
    target_cells = get_target_cells(targets)
    targets_by_id = {target.id: target for target in targets}

    # A partially intersected cell is not treated as fully observed for
    # negative evidence. However, a target whose true continuous position
    # lies inside the camera footprint must still be detectable even if its
    # grid-cell center lies just outside the footprint.
    target_cells_with_visible_target = set()
    for target in targets:
        if np.linalg.norm(target.position - uav.position[:2]) <= fov_radius:
            target_cells_with_visible_target.add(world_to_grid(target.position))

    observed_cells = sorted(
        center_visible_cells | target_cells_with_visible_target,
        key=lambda cell: (cell[1], cell[0]),
    )

    sensing_log = []

    for gx, gy in observed_cells:
        candidate_target_ids = target_cells.get((gx, gy), [])

        target_ids_in_fov = [
            target_id
            for target_id in candidate_target_ids
            if np.linalg.norm(
                targets_by_id[target_id].position - uav.position[:2]
            ) <= fov_radius
        ]

        has_target = len(target_ids_in_fov) > 0
        observation = sample_sensor_measurement(
            has_target,
            pd,
            pf,
            rng,
        )

        prior = float(belief_map[gy, gx])
        posterior = bayes_update(
            prior,
            observation,
            pd,
            pf,
        )

        belief_map[gy, gx] = posterior

        sensing_log.append({
            "gx": gx,
            "gy": gy,
            "target_ids": target_ids_in_fov,
            "has_target": has_target,
            "observation": observation,
            "prior": prior,
            "posterior": posterior,
            "pd": pd,
            "pf": pf,
        })

    return sensing_log

In [37]:
def check_confirmation(uav, sensing_record):
    posterior = float(sensing_record["posterior"])
    observation = int(sensing_record["observation"])
    target_ids = list(sensing_record["target_ids"])

    if posterior < CONFIG["confirmation_threshold"]:
        return "none", []

    if observation != 1:
        return "none", []

    if target_ids:
        return "true_confirmation", target_ids

    return "false_confirmation", []

In [38]:
@dataclass
class ConfirmationEvent:
    target_id: int | None
    uav_id: int
    step: int

    cell: tuple
    uav_position: np.ndarray

    altitude: float
    belief: float
    observation: int

    confirmation_type: str

In [39]:
def create_confirmation_events(
    uav,
    sensing_record,
    step
):
    confirmation_type, target_ids = check_confirmation(
        uav,
        sensing_record
    )

    if confirmation_type == "none":
        return []

    common = dict(
        uav_id=int(uav.id),
        step=int(step),

        cell=(
            int(sensing_record["gx"]),
            int(sensing_record["gy"]),
        ),

        uav_position=uav.position.copy(),

        altitude=float(uav.position[2]),
        belief=float(sensing_record["posterior"]),
        observation=int(sensing_record["observation"]),
        confirmation_type=confirmation_type,
    )

    if confirmation_type == "false_confirmation":
        return [ConfirmationEvent(target_id=None,**common,)]

    return [ConfirmationEvent(target_id=int(target_id),**common,) for target_id in target_ids]

In [40]:
def process_confirmation_events(
    events,
    targets,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
):
    reports = []
    event_log = []

    if not isinstance(report_buffers, list):
        raise TypeError("report_buffers must be a list")
    if not isinstance(pending_reports, list):
        raise TypeError("pending_reports must be a list")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")

    targets_by_id = {
        target.id: target
        for target in targets
    }

    generated_target_ids = set()

    for event in events:
        event_log.append(event)

        if event.confirmation_type != "true_confirmation":
            continue

        if event.target_id not in targets_by_id:
            raise ValueError("confirmation event target_id not found")

        target = targets_by_id[event.target_id]
        target.confirmed = True

        if target.id in gcs_received_target_ids:
            continue

        report_already_buffered = any(
            report.target_id == target.id
            for buffer in report_buffers
            for report in buffer
        )
        report_already_pending = any(
            report.target_id == target.id
            for report in pending_reports
        )

        if (
            report_already_buffered
            or report_already_pending
            or target.id in generated_target_ids
        ):
            continue

        source_uav = int(event.uav_id)
        if not 0 <= source_uav < len(report_buffers):
            raise ValueError("confirmation event uav_id out of range")

        report = Report(
            target_id=target.id,
            source_uav=source_uav,
            created_step=event.step,
            size_bytes=CONFIG["report_bytes"],
            ttl_s=CONFIG["report_ttl"],
        )

        enqueued, reason = enqueue_report(
            report_buffers[source_uav],
            report,
        )

        if not enqueued:
            if reason == "buffer_full":
                pending_reports.append(report)
            elif reason != "duplicate":
                raise RuntimeError(f"unexpected enqueue result: {reason}")

        reports.append(report)
        generated_target_ids.add(target.id)

    return reports, event_log

In [41]:
def create_report_buffers():
    return [[] for _ in range(CONFIG["num_uavs"])]


def create_pending_reports():
    return []


def create_gcs_received_target_ids():
    return set()

In [42]:
def report_remaining_bytes(report):
    remaining = report.size_bytes - report.delivered_bytes
    return max(0, int(remaining))


def report_is_complete(report):
    return report_remaining_bytes(report) == 0


def buffer_used_bytes(buffer):
    return sum(report.size_bytes for report in buffer)

In [43]:
def report_exists(buffer, target_id):
    return any(
        report.target_id == target_id for report in buffer
    )


def report_exists_in_buffers(report_buffers, target_id):
    return any(
        report_exists(buffer, target_id)
        for buffer in report_buffers
    )


def pending_report_exists(pending_reports, target_id):
    return any(
        report.target_id == target_id
        for report in pending_reports
    )

In [44]:
def enqueue_report(buffer, report):
    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")
    if not isinstance(report, Report):
        raise TypeError("report must be a Report")

    # Completed reports no longer occupy local buffer capacity.
    buffer[:] = [
        existing
        for existing in buffer
        if not report_is_complete(existing)
    ]

    if report_exists(buffer, report.target_id):
        return False, "duplicate"

    used_bytes = buffer_used_bytes(buffer)
    new_used_bytes = used_bytes + report.size_bytes

    if new_used_bytes > CONFIG["buffer_bytes"]:
        return False, "buffer_full"

    buffer.append(report)
    return True, "enqueued"

In [ ]:
def report_age_s(report, current_step):
    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(current_step, (int, np.integer))
    ):
        raise TypeError("current_step must be an integer")

    created_step = report.created_step

    if (
        isinstance(created_step, (bool, np.bool_))
        or not isinstance(created_step, (int, np.integer))
    ):
        raise TypeError("report.created_step must be an integer")

    current_step = int(current_step)
    created_step = int(created_step)

    if created_step < 0:
        raise ValueError("report.created_step must be >= 0")

    if current_step < created_step:
        raise ValueError(
            "current_step must be >= report.created_step"
        )

    dt = float(CONFIG["dt"])

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError("CONFIG['dt'] must be finite and > 0")

    return float(
        (current_step - created_step) * dt
    )


In [46]:
def report_is_expired(report, current_step):
    age_s = report_age_s(report,current_step)

    return age_s >= report.ttl_s

In [47]:
def remove_expired_reports(
    buffer,
    current_step,
):
    kept_reports = []
    expired_reports = []

    for report in buffer:
        if report_is_expired(report, current_step):
            expired_reports.append(report)
        else:
            kept_reports.append(report)

    buffer[:] = kept_reports
    return expired_reports


def remove_completed_reports(buffer):
    completed_reports = [
        report
        for report in buffer
        if report_is_complete(report)
    ]
    buffer[:] = [
        report
        for report in buffer
        if not report_is_complete(report)
    ]
    return completed_reports


def cleanup_report_buffer(buffer, current_step):
    expired = remove_expired_reports(buffer, current_step)
    completed = remove_completed_reports(buffer)
    return {
        "expired": expired,
        "completed": completed,
    }


def flush_pending_reports(
    pending_reports,
    report_buffers,
    current_step,
    gcs_received_target_ids,
):
    if not isinstance(pending_reports, list):
        raise TypeError("pending_reports must be a list")
    if not isinstance(report_buffers, list):
        raise TypeError("report_buffers must be a list")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")

    for buffer in report_buffers:
        cleanup_report_buffer(buffer, current_step)

    kept_pending = []
    enqueued_target_ids = []
    expired_target_ids = []
    discarded_target_ids = []

    for report in pending_reports:
        if report.target_id in gcs_received_target_ids:
            discarded_target_ids.append(report.target_id)
            continue

        if report_is_expired(report, current_step):
            expired_target_ids.append(report.target_id)
            continue

        if report_exists_in_buffers(report_buffers, report.target_id):
            discarded_target_ids.append(report.target_id)
            continue

        source_uav = int(report.source_uav)
        if not 0 <= source_uav < len(report_buffers):
            raise ValueError("pending report source_uav out of range")

        enqueued, reason = enqueue_report(
            report_buffers[source_uav],
            report,
        )

        if enqueued:
            enqueued_target_ids.append(report.target_id)
        elif reason == "buffer_full":
            kept_pending.append(report)
        elif reason == "duplicate":
            discarded_target_ids.append(report.target_id)
        else:
            raise RuntimeError(f"unexpected enqueue result: {reason}")

    pending_reports[:] = kept_pending

    return {
        "enqueued_target_ids": enqueued_target_ids,
        "expired_target_ids": expired_target_ids,
        "discarded_target_ids": discarded_target_ids,
    }


def mark_target_delivered_to_gcs(
    target_id,
    gcs_received_target_ids,
    report_buffers,
    pending_reports,
):
    if isinstance(target_id, (bool, np.bool_)) or not isinstance(
        target_id,
        (int, np.integer),
    ):
        raise TypeError("target_id must be an integer")

    target_id = int(target_id)
    if target_id < 0:
        raise ValueError("target_id must be >= 0")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")

    gcs_received_target_ids.add(target_id)

    removed_from_buffers = 0
    for buffer in report_buffers:
        before = len(buffer)
        buffer[:] = [
            report
            for report in buffer
            if report.target_id != target_id
        ]
        removed_from_buffers += before - len(buffer)

    before_pending = len(pending_reports)
    pending_reports[:] = [
        report
        for report in pending_reports
        if report.target_id != target_id
    ]

    return {
        "removed_from_buffers": removed_from_buffers,
        "removed_from_pending": before_pending - len(pending_reports),
    }

In [48]:
def distance_3d(position_a, position_b):
    position_a = np.asarray(position_a,dtype=np.float64)
    position_b = np.asarray(position_b, dtype=np.float64)

    if position_a.shape != (3,):
        raise ValueError("position_a must have shape (3,)")

    if position_b.shape != (3,):
        raise ValueError("position_b must have shape (3,)")

    if not np.all(np.isfinite(position_a)):
        raise ValueError("position_a must contain only finite values")

    if not np.all(np.isfinite(position_b)):
        raise ValueError("position_b must contain only finite values")

    return np.linalg.norm(position_a - position_b)
    

In [49]:
def positions_can_communicate(position_a,position_b,max_range_m):
    max_range_m = float(max_range_m)

    if (
        not np.isfinite(max_range_m)
        or max_range_m <= 0.0
    ):
        raise ValueError(
            "max_range_m must be finite and > 0"
        )

    return (distance_3d(position_a,position_b)<= max_range_m)

In [50]:
def uavs_can_communicate(uav_a,uav_b):
    if uav_a.id == uav_b.id:
        return False

    if not uav_a.active:
        return False

    if not uav_b.active:
        return False

    return positions_can_communicate(
        uav_a.position,
        uav_b.position,
        CONFIG["peer_contact_range_m"]
    )

In [51]:
def uav_can_reach_gcs(uav):
    if not uav.active:
        return False

    return positions_can_communicate(
        uav.position,
        CONFIG["gcs_position"],
        CONFIG["gcs_contact_range_m"],
    )

In [52]:
def get_uav_neighbors(uav,uavs):
    neighbors = []

    for other in uavs:
        if uavs_can_communicate(uav,other):
            neighbors.append(other.id)

    return neighbors

In [53]:
GCS_NODE = -1
SILENT_DESTINATION = None

In [54]:
def communication_choices(sender_id,num_uavs=None):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if (isinstance(sender_id, bool) or not isinstance(sender_id,(int, np.integer))):
        raise TypeError("sender_id must be an integer")

    if (isinstance(num_uavs, bool) or not isinstance(num_uavs,(int, np.integer))):
        raise TypeError("num_uavs must be an integer")

    sender_id = int(sender_id)
    num_uavs = int(num_uavs)

    if num_uavs < 1:
        raise ValueError("num_uavs must be >= 1")

    if not 0 <= sender_id < num_uavs:
        raise ValueError("sender_id out of range")

    peers = tuple(
        uav_id
        for uav_id in range(num_uavs)
        if uav_id != sender_id
    )

    return (SILENT_DESTINATION,*peers,GCS_NODE)

In [ ]:
def decode_destination(
    sender_id,
    destination_index,
    num_uavs=None,
):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if (
        isinstance(
            destination_index,
            (bool, np.bool_),
        )
        or not isinstance(
            destination_index,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "destination_index must be an integer"
        )

    destination_index = int(destination_index)

    choices = communication_choices(
        sender_id,
        num_uavs,
    )

    if not 0<= destination_index< len(choices):
        raise ValueError(
            "destination_index out of range"
        )

    return choices[destination_index]

In [ ]:
def decode_tx_power_w(power_action):
    power_array = np.asarray(power_action)

    if power_array.shape not in [(), (1,)]:
        raise ValueError(
            "power_action must be a scalar or have shape (1,)"
        )

    if (
        np.issubdtype(power_array.dtype, np.bool_)
        or not np.issubdtype(power_array.dtype, np.number)
    ):
        raise TypeError(
            "power_action must be numeric and not boolean"
        )

    power_value = float(power_array.item())

    if not np.isfinite(power_value):
        raise ValueError(
            "power_action must be finite"
        )

    if not -1.0 <= power_value <= 1.0:
        raise ValueError(
            "power_action must be within [-1, 1]"
        )

    power_min = float(CONFIG["tx_power_min_w"])
    power_max = float(CONFIG["tx_power_max_w"])

    if (
        not np.isfinite(power_min)
        or not np.isfinite(power_max)
        or power_min <= 0.0
        or power_max < power_min
    ):
        raise ValueError(
            "invalid TX power range"
        )

    normalized = (power_value + 1.0) / 2.0

    return float(
        power_min
        + normalized
        * (power_max - power_min)
    )


In [60]:
@dataclass
class HybridAction:
    movement: np.ndarray
    destination: int | None
    tx_power_w: float

In [ ]:
def decode_hybrid_action(
    sender_id,
    action,
    num_uavs=None,
):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if not isinstance(action, dict):
        raise TypeError(
            "action must be a dict"
        )

    required_keys = {
        "motion",
        "destination",
        "power",
    }

    if set(action.keys()) != required_keys:
        raise ValueError(
            "action must contain exactly "
            "motion, destination, power"
        )

    motion_object = np.asarray(
        action["motion"],
        dtype=object,
    )

    if motion_object.shape != (3,):
        raise ValueError(
            "motion must have shape (3,)"
        )

    for value in motion_object:
        if isinstance(value, (bool, np.bool_)):
            raise TypeError(
                "motion must be numeric and not boolean"
            )

        if not isinstance(
            value,
            (int, float, np.integer, np.floating),
        ):
            raise TypeError(
                "motion must be numeric and not boolean"
            )

    motion = motion_object.astype(
        np.float64,
    )

    if not np.all(np.isfinite(motion)):
        raise ValueError(
            "motion must contain only finite values"
        )

    if (
        np.any(motion < -1.0)
        or np.any(motion > 1.0)
    ):
        raise ValueError(
            "motion must be within [-1, 1]"
        )

    destination = decode_destination(
        sender_id,
        action["destination"],
        num_uavs,
    )

    decoded_power_w = decode_tx_power_w(action["power"])

    if destination is SILENT_DESTINATION:
        tx_power_w = 0.0
    else:
        tx_power_w = decoded_power_w

    return HybridAction(
        movement=motion.copy(),
        destination=destination,
        tx_power_w=tx_power_w,
    )


In [ ]:
def select_report_for_transmission(
    buffer,
    current_step,
):
    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(current_step, (int, np.integer))
    ):
        raise TypeError("current_step must be an integer")

    current_step = int(current_step)

    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")

    for report in buffer:
        if not isinstance(report, Report):
            raise TypeError("buffer must contain only Report objects")

    cleanup_report_buffer(buffer, current_step)

    if not buffer:
        return None

    return buffer[0]

In [ ]:
@dataclass(frozen=True)
class TransmissionIntent:
    sender: int
    recipient: int
    target_id: int
    requested_bytes: int
    tx_power_w: float

    def __post_init__(self):
        for name, value in (
            ("sender", self.sender),
            ("recipient", self.recipient),
            ("target_id", self.target_id),
            ("requested_bytes", self.requested_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(
                    value,
                    (int, np.integer),
                )
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        sender = int(self.sender)
        recipient = int(self.recipient)
        target_id = int(self.target_id)
        requested_bytes = int(
            self.requested_bytes
        )
        tx_power_w = float(
            self.tx_power_w
        )

        num_uavs = int(
            CONFIG["num_uavs"]
        )

        if not 0 <= sender < num_uavs:
            raise ValueError(
                "sender out of range"
            )

        if (
            recipient != GCS_NODE
            and not 0 <= recipient < num_uavs
        ):
            raise ValueError(
                "recipient must be GCS_NODE "
                "or a valid UAV id"
            )

        if recipient == sender:
            raise ValueError(
                "sender and recipient "
                "must be different"
            )

        if target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if requested_bytes <= 0:
            raise ValueError(
                "requested_bytes must be > 0"
            )

        power_min = float(
            CONFIG["tx_power_min_w"]
        )
        power_max = float(
            CONFIG["tx_power_max_w"]
        )

        if (
            not np.isfinite(tx_power_w)
            or not power_min
            <= tx_power_w
            <= power_max
        ):
            raise ValueError(
                "tx_power_w outside "
                "configured range"
            )

        object.__setattr__(
            self,
            "sender",
            sender,
        )
        object.__setattr__(
            self,
            "recipient",
            recipient,
        )
        object.__setattr__(
            self,
            "target_id",
            target_id,
        )
        object.__setattr__(
            self,
            "requested_bytes",
            requested_bytes,
        )
        object.__setattr__(
            self,
            "tx_power_w",
            tx_power_w,
        )

In [61]:
def build_transmission_intent(
    sender_id,
    hybrid_action,
    buffer,
    current_step,
):
    if (
        isinstance(
            sender_id,
            (bool, np.bool_),
        )
        or not isinstance(
            sender_id,
            (int, np.integer),
        )
    ):
        raise TypeError("sender_id must be an integer")

    sender_id = int(sender_id)

    if sender_id < 0:
        raise ValueError("sender_id must be >= 0")

    if not isinstance(
        hybrid_action,
        HybridAction,
    ):
        raise TypeError("hybrid_action must be HybridAction")

    if hybrid_action.destination is SILENT_DESTINATION:
        return None

    report = select_report_for_transmission(
        buffer,
        current_step,
    )

    if report is None:
        return None

    remaining_bytes = (
        report.size_bytes
        - report.delivered_bytes
    )

    if remaining_bytes <= 0:
        return None

    return TransmissionIntent(
        sender=sender_id,
        recipient=hybrid_action.destination,
        target_id=report.target_id,
        requested_bytes=remaining_bytes,
        tx_power_w=hybrid_action.tx_power_w,
    )


In [ ]:
def transmission_is_feasible(
    intent,
    uavs,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if len(uavs) != num_uavs:
        raise ValueError(
            "uavs length does not match "
            "CONFIG['num_uavs']"
        )

    for uav in uavs:
        if not isinstance(uav, UAV):
            raise TypeError(
                "uavs must contain only UAV objects"
            )

        if (
            isinstance(uav.id, (bool, np.bool_))
            or not isinstance(
                uav.id,
                (int, np.integer),
            )
        ):
            raise TypeError(
                "each UAV id must be an integer"
            )

    ids = [
        int(uav.id)
        for uav in uavs
    ]

    if len(set(ids)) != num_uavs:
        raise ValueError(
            "UAV ids must be unique"
        )

    if set(ids) != set(range(num_uavs)):
        raise ValueError(
            "UAV ids must be exactly "
            "0..num_uavs-1"
        )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
    }

    sender = uavs_by_id[
        intent.sender
    ]

    if not sender.active:
        return False

    if intent.recipient == GCS_NODE:
        return bool(
            uav_can_reach_gcs(
                sender
            )
        )

    recipient = uavs_by_id[
        intent.recipient
    ]

    return bool(
        uavs_can_communicate(
            sender,
            recipient,
        )
    )


In [ ]:
LOS_LINK = "los"
NLOS_LINK = "nlos"


def positions_have_los(
    position_a,
    position_b,
    obstacles,
):
    if obstacles is None:
        obstacles = []

    if not isinstance(obstacles, list):
        raise TypeError("obstacles must be a list")

    for obstacle in obstacles:
        if not isinstance(obstacle, Obstacle):
            raise TypeError(
                "obstacles must contain only Obstacle objects"
            )

    return not segment_intersects_obstacle(
        position_a,
        position_b,
        obstacles,
    )


def classify_link_state(
    position_a,
    position_b,
    obstacles,
):
    if positions_have_los(
        position_a,
        position_b,
        obstacles,
    ):
        return LOS_LINK

    return NLOS_LINK


In [ ]:
def db_to_linear(db_value):
    db_value = float(db_value)

    if not np.isfinite(db_value):
        raise ValueError(
            "db_value must be finite"
        )

    return float(
        10.0 ** (db_value / 10.0)
    )


def dbm_to_w(dbm_value):
    dbm_value = float(dbm_value)

    if not np.isfinite(dbm_value):
        raise ValueError(
            "dbm_value must be finite"
        )

    return float(
        10.0 ** (
            (dbm_value - 30.0) / 10.0
        )
    )


def calculate_link_snr(
    tx_power_w,
    distance_m,
    additional_loss_db=0.0,
):
    tx_power_w = float(tx_power_w)
    distance_m = float(distance_m)
    additional_loss_db = float(additional_loss_db)

    if (
        not np.isfinite(tx_power_w)
        or tx_power_w <= 0.0
    ):
        raise ValueError(
            "tx_power_w must be finite and > 0"
        )

    if (
        not np.isfinite(distance_m)
        or distance_m < 0.0
    ):
        raise ValueError(
            "distance_m must be finite and >= 0"
        )

    if (
        not np.isfinite(additional_loss_db)
        or additional_loss_db < 0.0
    ):
        raise ValueError(
            "additional_loss_db must be finite and >= 0"
        )

    power_min = float(
        CONFIG["tx_power_min_w"]
    )
    power_max = float(
        CONFIG["tx_power_max_w"]
    )

    if not (
        power_min
        <= tx_power_w
        <= power_max
    ):
        raise ValueError(
            "tx_power_w outside configured range"
        )

    reference_distance_m = float(
        CONFIG["comm_reference_distance_m"]
    )
    path_loss_exponent = float(
        CONFIG["comm_path_loss_exponent"]
    )

    if (
        not np.isfinite(reference_distance_m)
        or reference_distance_m <= 0.0
    ):
        raise ValueError(
            "comm_reference_distance_m "
            "must be finite and > 0"
        )

    if (
        not np.isfinite(path_loss_exponent)
        or path_loss_exponent <= 0.0
    ):
        raise ValueError(
            "comm_path_loss_exponent "
            "must be finite and > 0"
        )

    reference_gain_linear = db_to_linear(
        CONFIG["comm_reference_gain_db"]
    )

    noise_power_w = dbm_to_w(
        CONFIG["comm_noise_power_dbm"]
    )

    if (
        reference_gain_linear <= 0.0
        or noise_power_w <= 0.0
    ):
        raise ValueError(
            "invalid communication model"
        )

    effective_distance_m = max(
        distance_m,
        reference_distance_m,
    )

    channel_power_gain = (
        reference_gain_linear
        * (
            reference_distance_m
            / effective_distance_m
        )
        ** path_loss_exponent
    )

    additional_gain = db_to_linear(
        -additional_loss_db
    )

    received_power_w = (
        tx_power_w
        * channel_power_gain
        * additional_gain
    )

    snr_linear = (
        received_power_w
        / noise_power_w
    )

    return float(snr_linear)


def snr_linear_to_db(snr_linear):
    snr_linear = float(snr_linear)

    if (
        not np.isfinite(snr_linear)
        or snr_linear <= 0.0
    ):
        raise ValueError(
            "snr_linear must be finite and > 0"
        )

    return float(
        10.0 * np.log10(snr_linear)
    )


In [ ]:
def transmission_distance_m(
    intent,
    uavs,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
        if isinstance(uav, UAV)
    }

    if intent.sender not in uavs_by_id:
        raise ValueError(
            "sender UAV not found"
        )

    sender = uavs_by_id[
        intent.sender
    ]

    if intent.recipient == GCS_NODE:
        return distance_3d(
            sender.position,
            CONFIG["gcs_position"],
        )

    if intent.recipient not in uavs_by_id:
        raise ValueError(
            "recipient UAV not found"
        )

    recipient = uavs_by_id[
        intent.recipient
    ]

    return distance_3d(
        sender.position,
        recipient.position,
    )


def calculate_intent_snr(
    intent,
    uavs,
    obstacles=None,
):
    if obstacles is None:
        obstacles = []

    if not transmission_is_feasible(
        intent,
        uavs,
    ):
        return 0.0

    distance_m = transmission_distance_m(
        intent,
        uavs,
    )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
        if isinstance(uav, UAV)
    }

    sender_position = uavs_by_id[
        intent.sender
    ].position

    if intent.recipient == GCS_NODE:
        recipient_position = np.asarray(
            CONFIG["gcs_position"],
            dtype=np.float64,
        )
    else:
        recipient_position = uavs_by_id[
            intent.recipient
        ].position

    link_state = classify_link_state(
        sender_position,
        recipient_position,
        obstacles,
    )

    if link_state == LOS_LINK:
        additional_loss_db = 0.0
    else:
        additional_loss_db = float(
            CONFIG["comm_nlos_additional_loss_db"]
        )

    return calculate_link_snr(
        intent.tx_power_w,
        distance_m,
        additional_loss_db=additional_loss_db,
    )


In [ ]:
# Regression: current world configuration and GCS-independent launch sampling.
_reg_rng, _reg_uavs, _reg_targets, _reg_obstacles = create_world(44)

assert len(_reg_uavs) == CONFIG["num_uavs"]
assert len(_reg_targets) == CONFIG["num_targets"]
assert len(_reg_obstacles) == CONFIG["num_obstacles"]

_reg_gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
_reg_launch_radius = float(CONFIG["launch_radius_m"])

for _uav in _reg_uavs:
    assert 0.0 <= _uav.position[0] <= CONFIG["map_size"]
    assert 0.0 <= _uav.position[1] <= CONFIG["map_size"]
    assert _uav.position[2] == float(CONFIG["altitude_min"])
    assert (
        np.linalg.norm(_uav.position[:2] - _reg_gcs_xy)
        <= _reg_launch_radius + 1e-9
    )

for _i in range(len(_reg_uavs)):
    for _j in range(_i + 1, len(_reg_uavs)):
        _distance = np.linalg.norm(
            _reg_uavs[_i].position - _reg_uavs[_j].position
        )
        assert _distance >= CONFIG["launch_min_spacing_m"]

# Same seed -> same UAV launch layout.
_reg_uavs_same_seed = create_uav(np.random.default_rng(44))
assert np.allclose(
    np.stack([_u.position for _u in _reg_uavs]),
    np.stack([_u.position for _u in _reg_uavs_same_seed]),
)

# Launch geometry must not depend on a preferred map direction.
_reg_original_gcs = list(CONFIG["gcs_position"])
_reg_original_num_uavs = int(CONFIG["num_uavs"])

try:
    _reg_gcs_cases = [
        [0.0, 0.0, 0.0],
        [2500.0, 0.0, 0.0],
        [5000.0, 0.0, 0.0],
        [0.0, 2500.0, 0.0],
        [2500.0, 2500.0, 0.0],
        [5000.0, 2500.0, 0.0],
        [0.0, 5000.0, 0.0],
        [2500.0, 5000.0, 0.0],
        [5000.0, 5000.0, 0.0],
    ]

    for _reg_gcs in _reg_gcs_cases:
        CONFIG["gcs_position"] = _reg_gcs
        CONFIG["num_uavs"] = 20
        _reg_case_uavs = create_uav(np.random.default_rng(44))
        _reg_case_gcs_xy = np.asarray(_reg_gcs[:2], dtype=np.float64)

        assert len(_reg_case_uavs) == 20

        for _uav in _reg_case_uavs:
            assert 0.0 <= _uav.position[0] <= CONFIG["map_size"]
            assert 0.0 <= _uav.position[1] <= CONFIG["map_size"]
            assert (
                np.linalg.norm(_uav.position[:2] - _reg_case_gcs_xy)
                <= CONFIG["launch_radius_m"] + 1e-9
            )

        for _i in range(len(_reg_case_uavs)):
            for _j in range(_i + 1, len(_reg_case_uavs)):
                assert (
                    np.linalg.norm(
                        _reg_case_uavs[_i].position
                        - _reg_case_uavs[_j].position
                    )
                    >= CONFIG["launch_min_spacing_m"]
                )
finally:
    CONFIG["gcs_position"] = _reg_original_gcs
    CONFIG["num_uavs"] = _reg_original_num_uavs


# Regression: LoS/NLoS obstacle-aware communication.
_reg_blocking_obstacle = Obstacle(
    position=np.array([50.0, 0.0], dtype=np.float64),
    radius=10.0,
    height=100.0,
)

assert segment_intersects_obstacle(
    np.array([0.0, 0.0, 50.0]),
    np.array([100.0, 0.0, 50.0]),
    [_reg_blocking_obstacle],
)
assert not segment_intersects_obstacle(
    np.array([0.0, 0.0, 150.0]),
    np.array([100.0, 0.0, 150.0]),
    [_reg_blocking_obstacle],
)

assert classify_link_state(
    np.array([0.0, 0.0, 50.0]),
    np.array([100.0, 0.0, 50.0]),
    [],
) == LOS_LINK
assert classify_link_state(
    np.array([0.0, 0.0, 50.0]),
    np.array([100.0, 0.0, 50.0]),
    [_reg_blocking_obstacle],
) == NLOS_LINK

_reg_los_snr = calculate_link_snr(
    0.1,
    1000.0,
    additional_loss_db=0.0,
)
_reg_nlos_snr = calculate_link_snr(
    0.1,
    1000.0,
    additional_loss_db=CONFIG["comm_nlos_additional_loss_db"],
)
assert np.isclose(_reg_los_snr, 1.0)
assert 0.0 < _reg_nlos_snr < _reg_los_snr
assert np.isclose(
    _reg_nlos_snr,
    _reg_los_snr
    * 10.0 ** (-CONFIG["comm_nlos_additional_loss_db"] / 10.0),
)


# Regression: Obstacle.position has one canonical representation: XY center.
try:
    Obstacle(
        position=np.array([50.0, 0.0, 0.0], dtype=np.float64),
        radius=10.0,
        height=100.0,
    )
except ValueError:
    pass
else:
    raise AssertionError("3D Obstacle.position must be rejected")

# Regression: confirmation/report lifecycle survives buffer pressure without re-confirmation.
_reg_target = Target(
    id=0,
    position=np.array([100.0, 100.0], dtype=np.float64),
)
_reg_confirmation = ConfirmationEvent(
    target_id=0,
    uav_id=0,
    step=10,
    cell=(4, 4),
    uav_position=np.array([100.0, 100.0, 50.0], dtype=np.float64),
    altitude=50.0,
    belief=0.999,
    observation=1,
    confirmation_type="true_confirmation",
)
_reg_buffers = create_report_buffers()
_reg_pending = create_pending_reports()
_reg_received = create_gcs_received_target_ids()
_reg_buffers[0][:] = [
    Report(
        target_id=target_id,
        source_uav=0,
        created_step=0,
        size_bytes=CONFIG["report_bytes"],
        ttl_s=CONFIG["report_ttl"],
    )
    for target_id in (1, 2, 3)
]
_reg_reports, _ = process_confirmation_events(
    [_reg_confirmation],
    [_reg_target],
    _reg_buffers,
    _reg_pending,
    _reg_received,
)
assert _reg_target.confirmed
assert len(_reg_reports) == 1
assert [report.target_id for report in _reg_pending] == [0]

# No second observation is required. Once capacity is available, pending data is queued.
_reg_buffers[0].clear()
_reg_flush = flush_pending_reports(
    _reg_pending,
    _reg_buffers,
    current_step=11,
    gcs_received_target_ids=_reg_received,
)
assert _reg_flush["enqueued_target_ids"] == [0]
assert _reg_pending == []
assert [report.target_id for report in _reg_buffers[0]] == [0]

# GCS delivery is a separate mission/network state and suppresses future duplicates.
mark_target_delivered_to_gcs(
    0,
    _reg_received,
    _reg_buffers,
    _reg_pending,
)
assert 0 in _reg_received
assert not report_exists_in_buffers(_reg_buffers, 0)
_reg_after_delivery, _ = process_confirmation_events(
    [_reg_confirmation],
    [_reg_target],
    _reg_buffers,
    _reg_pending,
    _reg_received,
)
assert _reg_after_delivery == []

# Regression: a completed FIFO head is cleaned up and cannot block later reports.
_reg_completed = Report(
    target_id=1,
    source_uav=0,
    created_step=0,
    size_bytes=1_000_000,
    ttl_s=300.0,
    delivered_bytes=1_000_000,
)
_reg_next = Report(
    target_id=2,
    source_uav=0,
    created_step=0,
    size_bytes=1_000_000,
    ttl_s=300.0,
)
_reg_fifo = [_reg_completed, _reg_next]
assert select_report_for_transmission(_reg_fifo, current_step=1).target_id == 2
assert [report.target_id for report in _reg_fifo] == [2]

# Regression: a partial report requests only its remaining bytes.
_reg_partial_report = Report(
    target_id=0,
    source_uav=0,
    created_step=0,
    size_bytes=1_000_000,
    ttl_s=300.0,
    delivered_bytes=700_000,
)
_reg_partial_action = HybridAction(
    movement=np.zeros(3, dtype=np.float64),
    destination=GCS_NODE,
    tx_power_w=float(CONFIG["tx_power_min_w"]),
)
_reg_partial_intent = build_transmission_intent(
    sender_id=0,
    hybrid_action=_reg_partial_action,
    buffer=[_reg_partial_report],
    current_step=1,
)
assert _reg_partial_intent.requested_bytes == 300_000

# Regression: partial FOV overlap must not clear unseen portions of a belief cell.
_reg_grid_n = int(np.ceil(CONFIG["map_size"] / CONFIG["grid_cell_m"]))
_reg_belief = np.full((_reg_grid_n, _reg_grid_n), 0.5, dtype=np.float64)
_reg_low_uav = UAV(
    id=0,
    position=np.array([0.0, 0.0, 1.0], dtype=np.float64),
    velocity=np.zeros(3, dtype=np.float64),
    battery_j=CONFIG["battery_j"],
)
_reg_far_same_cell_target = Target(
    id=0,
    position=np.array([24.0, 24.0], dtype=np.float64),
)
sense_and_update(
    _reg_low_uav,
    _reg_belief,
    [_reg_far_same_cell_target],
    np.random.default_rng(123),
)
_reg_gx, _reg_gy = world_to_grid(_reg_far_same_cell_target.position)
assert np.isclose(_reg_belief[_reg_gy, _reg_gx], 0.5)

# But a target truly inside the continuous camera footprint remains detectable.
class _RegressionZeroRng:
    def random(self):
        return 0.0

_reg_belief2 = np.full((_reg_grid_n, _reg_grid_n), 0.5, dtype=np.float64)
_reg_near_target = Target(
    id=0,
    position=np.array([0.5, 0.5], dtype=np.float64),
)
_reg_logs = sense_and_update(
    _reg_low_uav,
    _reg_belief2,
    [_reg_near_target],
    _RegressionZeroRng(),
)
_reg_gx, _reg_gy = world_to_grid(_reg_near_target.position)
assert any(record["target_ids"] == [0] for record in _reg_logs)
assert _reg_belief2[_reg_gy, _reg_gx] > 0.5

# Regression: inactive UAV state must not retain stale non-zero velocity.
_reg_inactive = UAV(
    id=0,
    position=np.array([10.0, 10.0, 10.0], dtype=np.float64),
    velocity=np.array([7.0, 0.0, 0.0], dtype=np.float64),
    battery_j=CONFIG["battery_j"],
    active=False,
)
_reg_inactive_position = _reg_inactive.position.copy()
apply_swarm_motion(
    [_reg_inactive],
    np.zeros((1, 3), dtype=np.float64),
    [],
)
assert np.allclose(_reg_inactive.position, _reg_inactive_position)
assert np.allclose(_reg_inactive.velocity, np.zeros(3))
